In [35]:
# https://platform.olimpiada-ai.ro/ro/problems/188

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

In [36]:
train = pd.read_csv("/kaggle/input/datasets/abukanabek/fire-distinguisher/train.csv")
test = pd.read_csv("/kaggle/input/datasets/abukanabek/fire-distinguisher/test.csv")

train.shape, test.shape

((3232, 17), (800, 16))

In [37]:
train.head()

,id,gas_raw,temp_c,flame,gas_avg,temp_avg,dGas,dTemp,sensor_noise,timestamp_ms,sensor_id,battery_level,vehicle_speed,weather_condition,gas_sensor_drift,ambient_temp,label
0,7621,422.194432,118.043366,1,767.789333,NaN,-55.906124,2.801129,NaN,1212189282,S001,NaN,123.645528,rainy,362.321329,102.058677,FIRE
1,6224,652.020520,121.762018,1,-116.321698,117.187637,-25.912647,1.523781,-0.446243,1948644533,S003,94.640768,72.397748,rainy,518.720369,123.267611,FIRE
2,6584,248.605877,6.377835,0,275.077483,NaN,NaN,NaN,-0.920522,1403281804,S004,85.680628,87.808526,sunny,289.947889,0.763587,NORMAL
3,7845,872.419262,111.909726,1,865.917087,234.145157,-31.096726,0.779816,1.205554,1969672422,S001,102.969677,115.677446,rainy,927.954621,114.443297,FIRE
4,6567,1919.594960,NaN,0,NaN,NaN,NaN,-1.301399,-0.779358,1844367040,S004,NaN,71.741976,sunny,2193.352397,NaN,GAS_LEAK


In [38]:
train['label'].value_counts()

label
FIRE        815
NORMAL      812
GAS_LEAK    806
OVERHEAT    799
Name: count, dtype: int64

In [39]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3232 entries, 0 to 3231
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 3232 non-null   int64  
 1   gas_raw            2325 non-null   float64
 2   temp_c             2322 non-null   float64
 3   flame              3232 non-null   int64  
 4   gas_avg            2340 non-null   float64
 5   temp_avg           2327 non-null   float64
 6   dGas               2263 non-null   float64
 7   dTemp              2267 non-null   float64
 8   sensor_noise       2324 non-null   float64
 9   timestamp_ms       3232 non-null   int64  
 10  sensor_id          3232 non-null   object 
 11  battery_level      2338 non-null   float64
 12  vehicle_speed      2330 non-null   float64
 13  weather_condition  3232 non-null   object 
 14  gas_sensor_drift   2325 non-null   float64
 15  ambient_temp       2322 non-null   float64
 16  label              3232 

In [40]:
subtask1 = []
percentages_dict = train['label'].value_counts(normalize=True).map(lambda x: round(100*x, 2)).to_dict()

for label_type in ['NORMAL', 'GAS_LEAK', 'OVERHEAT', 'FIRE']:
    subtask1.append(percentages_dict[label_type])

temperatures_dict = train.groupby('label')['temp_c'].mean().map(lambda x: round(x, 1)).to_dict()

for label_type in ['NORMAL', 'GAS_LEAK', 'OVERHEAT', 'FIRE']:
    subtask1.append(temperatures_dict[label_type])

gas_dict = train.groupby('label')['gas_raw'].mean().map(lambda x: round(x, 1)).to_dict()

for label_type in ['NORMAL', 'GAS_LEAK', 'OVERHEAT', 'FIRE']:
    subtask1.append(gas_dict[label_type])

len(subtask1)

12

In [41]:
median_fill = train.select_dtypes('number').median()
train.fillna(median_fill, inplace=True)
test.fillna(median_fill, inplace=True)
train.head()

,id,gas_raw,temp_c,flame,gas_avg,temp_avg,dGas,dTemp,sensor_noise,timestamp_ms,sensor_id,battery_level,vehicle_speed,weather_condition,gas_sensor_drift,ambient_temp,label
0,7621,422.194432,118.043366,1,767.789333,63.636718,-55.906124,2.801129,0.088936,1212189282,S001,60.006369,123.645528,rainy,362.321329,102.058677,FIRE
1,6224,652.020520,121.762018,1,-116.321698,117.187637,-25.912647,1.523781,-0.446243,1948644533,S003,94.640768,72.397748,rainy,518.720369,123.267611,FIRE
2,6584,248.605877,6.377835,0,275.077483,63.636718,1.106763,-0.004388,-0.920522,1403281804,S004,85.680628,87.808526,sunny,289.947889,0.763587,NORMAL
3,7845,872.419262,111.909726,1,865.917087,234.145157,-31.096726,0.779816,1.205554,1969672422,S001,102.969677,115.677446,rainy,927.954621,114.443297,FIRE
4,6567,1919.594960,73.652402,0,454.164968,63.636718,1.106763,-1.301399,-0.779358,1844367040,S004,60.006369,71.741976,sunny,2193.352397,69.570978,GAS_LEAK


In [42]:
from sklearn.preprocessing import StandardScaler

num_features = train.select_dtypes('number').columns.tolist()
num_features.remove('id')

scaler = StandardScaler()

train[num_features] = scaler.fit_transform(train[num_features])
test[num_features] = scaler.transform(test[num_features])

train.head()

,id,gas_raw,temp_c,flame,gas_avg,temp_avg,dGas,dTemp,sensor_noise,timestamp_ms,sensor_id,battery_level,vehicle_speed,weather_condition,gas_sensor_drift,ambient_temp,label
0,7621,-0.516032,1.038824,1.733481,0.664051,-0.097837,-2.491848,2.084890,-0.079294,-1.005063,S001,-0.035682,1.478208,rainy,-0.637759,0.705772,FIRE
1,6224,0.151169,1.121717,1.733481,-1.989695,1.104736,-1.176330,1.128621,-0.557417,1.584619,S003,1.220068,0.229861,rainy,-0.196957,1.176145,FIRE
2,6584,-1.019971,-1.450335,-0.576874,-0.814872,-0.097837,0.008745,-0.015421,-0.981134,-0.333102,S004,0.895197,0.605253,sunny,-0.841739,-1.540755,NORMAL
3,7845,0.791001,0.902097,1.733481,0.958591,3.731207,-1.403705,0.571662,0.918281,1.658561,S001,1.522051,1.284113,rainy,0.956446,0.980439,FIRE
4,6567,3.831023,0.049296,-0.576874,-0.277323,-0.097837,0.008745,-0.986410,-0.855019,1.217936,S004,-0.035682,0.213887,sunny,4.522895,-0.014742,GAS_LEAK


In [43]:
train['label_encoded'] = train['label'].map(dict([('NORMAL', 0), ('GAS_LEAK', 1), ('OVERHEAT', 2), ('FIRE', 3)]))

chosen_features = train[num_features + ['label_encoded']].corr(method='spearman')['label_encoded'].sort_values(ascending=False).index[1:1+6].tolist()

In [44]:
normal_mean = train[train['label']=='NORMAL'][chosen_features].mean().values
sigma = np.cov(train[train['label']=='NORMAL'][chosen_features].T)
sigma += np.identity(sigma.shape[0]) * 1e-6
sigma_inv = np.linalg.inv(sigma)
sigma_inv.shape

(6, 6)

In [45]:
from scipy.spatial.distance import mahalanobis

subtask2 = []

for idx, row in test[chosen_features].iterrows():
    dist = mahalanobis(row.values, normal_mean, sigma_inv)
    subtask2.append(round(dist, 2))

len(subtask2)

800

In [46]:
train_dists = []

for idx, row in train[chosen_features].iterrows():
    dist = mahalanobis(row.values, normal_mean, sigma_inv)
    train_dists.append(dist)

train_dists = np.array(train_dists)
p33, p66 = np.quantile(train_dists, 0.33), np.quantile(train_dists, 0.66)
p33, p66

(np.float64(1.7500010322574717), np.float64(3.862170150090393))

In [47]:
subtask3 = []

for val in subtask2:
    if val < p33:
        subtask3.append('LOW')
    elif val >= p66:
        subtask3.append('HIGH')
    else:
        subtask3.append('MEDIUM')

len(subtask3)

800

In [50]:
train = pd.read_csv("/kaggle/input/datasets/abukanabek/fire-distinguisher/train.csv")
test = pd.read_csv("/kaggle/input/datasets/abukanabek/fire-distinguisher/test.csv")

train.shape, test.shape

((3232, 17), (800, 16))

In [52]:
from sklearn.model_selection import train_test_split

features = [c for c in train.columns if c not in ['id', 'label']]
cat_features = [c for c in train.select_dtypes('object') if c in features]
target_col = 'label'

X, y = train[features], train[target_col]
X_test = test[features]

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train.shape, X_valid.shape

((2585, 15), (647, 15))

In [53]:
from catboost import Pool

train_pool = Pool(X_train, y_train, cat_features=cat_features)
valid_pool = Pool(X_valid, y_valid, cat_features=cat_features)

In [57]:
from catboost import CatBoostClassifier

params = {
    'iterations': 1000,
    'loss_function': 'MultiClass',
    'eval_metric': 'TotalF1:average=Weighted',
    'metric_period': 100,
    'random_state': 42,
    'max_depth': 3,
    'learning_rate': 0.1
}

model = CatBoostClassifier(**params)

model.fit(train_pool, eval_set=valid_pool)

0:	learn: 0.8149597	test: 0.8113135	best: 0.8113135 (0)	total: 6.42ms	remaining: 6.41s
100:	learn: 0.9631224	test: 0.9464035	best: 0.9464035 (100)	total: 439ms	remaining: 3.91s
200:	learn: 0.9742498	test: 0.9570382	best: 0.9570382 (200)	total: 842ms	remaining: 3.35s
300:	learn: 0.9796182	test: 0.9585076	best: 0.9585076 (300)	total: 1.27s	remaining: 2.96s
400:	learn: 0.9822963	test: 0.9630684	best: 0.9630684 (400)	total: 1.68s	remaining: 2.5s
500:	learn: 0.9857419	test: 0.9646169	best: 0.9646169 (500)	total: 2.07s	remaining: 2.06s
600:	learn: 0.9876532	test: 0.9646221	best: 0.9646221 (600)	total: 2.46s	remaining: 1.63s
700:	learn: 0.9891909	test: 0.9630690	best: 0.9646221 (600)	total: 2.84s	remaining: 1.21s
800:	learn: 0.9918897	test: 0.9661652	best: 0.9661652 (800)	total: 3.23s	remaining: 802ms
900:	learn: 0.9934344	test: 0.9646221	best: 0.9661652 (800)	total: 3.61s	remaining: 397ms
999:	learn: 0.9957465	test: 0.9615537	best: 0.9661652 (800)	total: 4s	remaining: 0us

bestTest = 0.96616

In [59]:
subtask4 = model.predict(X_test).flatten().tolist()
subtask4[:5]

['OVERHEAT', 'GAS_LEAK', 'OVERHEAT', 'FIRE', 'FIRE']

In [61]:
subm = pd.DataFrame({
    'subtaskID': [1] * 12 + [2] * len(test) + [3] * len(test) + [4] * len(test),
    'datapointID': list(range(1, 13)) + test['id'].tolist() * 3,
    'answer': subtask1 + subtask2 + subtask3 + subtask4
})

subm.to_csv("submission.csv", index=False)

subm

,subtaskID,datapointID,answer
0,1,1,25.12
1,1,2,24.94
2,1,3,24.72
3,1,4,25.22
4,1,5,28.5
...,...,...,...
2407,4,8600,FIRE
2408,4,8402,GAS_LEAK
2409,4,6744,OVERHEAT
2410,4,7891,GAS_LEAK
